# Hybrid Tables in Snowflake

Use hybrid tables for application-facing point reads, indexed lookups, and short transactions. This lab creates related customer and order tables, demonstrates enforced constraints, and practices commit and rollback.

Copy the SQL blocks into a Snowsight worksheet and run them in order. Select an existing writable database and an authorized role. The lab creates the `HybridTable` schema and uses `SNOWFLAKE_LEARNING_WH`.

## 1. How hybrid tables work

| Capability | Purpose |
|---|---|
| Primary row store | Efficient small-row reads and updates |
| Mandatory primary key | Identifies each row; uniqueness is enforced |
| Unique and foreign keys | Enforce supported integrity rules |
| Secondary indexes | Support selective lookups beyond the primary key |
| Row-level locking | Coordinate concurrent transactional updates |
| Warehouse compute | Executes queries and DML |

A hybrid table stores mutable operational records. Use the enforced keys, access patterns, and transaction boundaries together when designing the application.

[Hybrid table overview](https://docs.snowflake.com/en/user-guide/tables-hybrid)

## 2. Select the database and create the schema

Snowflake's hybrid-table limitations documentation lists AWS and Azure commercial-region availability and excludes trial accounts, Google Cloud, and SnowGov. Use an eligible account for this exercise; a different warehouse or schema will not remove an account restriction. [Hybrid-table availability](https://docs.snowflake.com/en/user-guide/tables-hybrid-limitations)

Select an existing **permanent, writable database** in Snowsight and a role with schema/table creation privileges and usage on `SNOWFLAKE_LEARNING_WH`. Hybrid tables cannot be created in transient databases or schemas. Keep this worksheet context throughout.

```sql
CREATE SCHEMA HybridTable;
USE SCHEMA HybridTable;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE(),
       CURRENT_ROLE(), CURRENT_REGION();
```

Replace `HybridTable` consistently with your team's schema name if needed. Unquoted schema names appear in uppercase. Run setup once in a new schema.

## 3. Storage, indexes, and constraints

A write goes directly to the row store and synchronously maintains relevant indexes. Snowflake asynchronously copies data to columnar object storage so larger scans do not have to disrupt operational access. The optimizer chooses a row-based or column-based scan. Hybrid storage is commonly larger than standard-table storage because row data compresses less efficiently.

The primary key is mandatory. Primary, unique, and foreign keys are enforced, unlike informational keys on standard Snowflake tables.

## 4. Create related hybrid tables and indexes

The customer table enforces unique email. The order table enforces its parent customer and includes a covering secondary index for status lookups.

```sql
USE SCHEMA HybridTable;

CREATE HYBRID TABLE APP_CUSTOMERS (
  CUSTOMER_ID INTEGER PRIMARY KEY,
  EMAIL VARCHAR(255) NOT NULL UNIQUE,
  CUSTOMER_NAME VARCHAR(100) NOT NULL,
  LOYALTY_POINTS INTEGER NOT NULL DEFAULT 0,
  UPDATED_AT TIMESTAMP_NTZ NOT NULL DEFAULT CURRENT_TIMESTAMP(),
  INDEX IDX_CUSTOMER_NAME (CUSTOMER_NAME)
);

CREATE HYBRID TABLE APP_ORDERS (
  ORDER_ID INTEGER PRIMARY KEY,
  CUSTOMER_ID INTEGER NOT NULL,
  ORDER_TS TIMESTAMP_NTZ NOT NULL,
  STATUS VARCHAR(20) NOT NULL,
  AMOUNT NUMBER(12,2) NOT NULL,
  CONSTRAINT FK_ORDER_CUSTOMER FOREIGN KEY (CUSTOMER_ID)
    REFERENCES APP_CUSTOMERS (CUSTOMER_ID),
  INDEX IDX_ORDER_STATUS (STATUS) INCLUDE (CUSTOMER_ID, AMOUNT)
);

SHOW HYBRID TABLES IN SCHEMA HybridTable;
SHOW INDEXES IN SCHEMA HybridTable;
```

## 5. Insert, query, update, and delete

Operational applications usually change a small number of rows per statement. Snowflake recommends `INSERT`, `UPDATE`, and `DELETE` for that pattern instead of `MERGE`.

```sql
INSERT INTO APP_CUSTOMERS
  (CUSTOMER_ID, EMAIL, CUSTOMER_NAME, LOYALTY_POINTS) VALUES
  (1, 'asha@example.com',  'Asha Rao',  100),
  (2, 'bilal@example.com', 'Bilal Khan', 40),
  (3, 'charu@example.com',  'Charu Sen',  20);

INSERT INTO APP_ORDERS VALUES
  (3001, 1, '2026-09-07 10:00:00', 'PLACED',  4500.00),
  (3002, 2, '2026-09-07 10:02:00', 'PLACED',  2200.00);

SELECT * FROM APP_CUSTOMERS WHERE CUSTOMER_ID = 1;
SELECT ORDER_ID, CUSTOMER_ID, AMOUNT
FROM APP_ORDERS WHERE STATUS = 'PLACED';

BEGIN;
UPDATE APP_CUSTOMERS
SET LOYALTY_POINTS = LOYALTY_POINTS + 85, UPDATED_AT = CURRENT_TIMESTAMP()
WHERE CUSTOMER_ID = 1;

INSERT INTO APP_ORDERS
VALUES (3003, 1, CURRENT_TIMESTAMP(), 'PLACED', 8500.00);
COMMIT;

UPDATE APP_ORDERS SET STATUS = 'SHIPPED' WHERE ORDER_ID = 3001;
DELETE FROM APP_ORDERS WHERE ORDER_ID = 3002;

SELECT C.CUSTOMER_ID, C.CUSTOMER_NAME, C.LOYALTY_POINTS,
       O.ORDER_ID, O.STATUS, O.AMOUNT
FROM APP_CUSTOMERS AS C
LEFT JOIN APP_ORDERS AS O ON O.CUSTOMER_ID = C.CUSTOMER_ID
ORDER BY C.CUSTOMER_ID, O.ORDER_ID;
```

Expected after the transaction and deletion: three customers, orders 3001 and 3003, and 185 loyalty points for customer 1.

## 6. Check constraint enforcement

Run each commented statement separately if you want to observe the error. Neither statement should add a row.

```sql
-- Duplicate primary key: expect a constraint violation.
-- INSERT INTO APP_CUSTOMERS
-- VALUES (1, 'another@example.com', 'Duplicate ID', 0, CURRENT_TIMESTAMP());

-- Missing parent customer: expect a foreign-key violation.
-- INSERT INTO APP_ORDERS
-- VALUES (3999, 999, CURRENT_TIMESTAMP(), 'PLACED', 10.00);

SELECT COUNT(*) AS CUSTOMER_1_COUNT
FROM APP_CUSTOMERS WHERE CUSTOMER_ID = 1;
SELECT COUNT(*) AS INVALID_PARENT_COUNT
FROM APP_ORDERS WHERE CUSTOMER_ID = 999;
```

Use hybrid tables for operational state, workflow metadata, application APIs, and low-latency lookup/update patterns. Standard tables remain the default for large analytical fact data. Review quotas, data-size limits, unsupported features, and regional availability before production use: [hybrid limitations](https://docs.snowflake.com/en/user-guide/tables-hybrid-limitations).

## 7. Roll back an uncommitted change

```sql
BEGIN TRANSACTION;
UPDATE APP_CUSTOMERS
SET LOYALTY_POINTS = LOYALTY_POINTS + 10
WHERE CUSTOMER_ID = 1;
SELECT CUSTOMER_ID, LOYALTY_POINTS
FROM APP_CUSTOMERS WHERE CUSTOMER_ID = 1;
ROLLBACK;
SELECT CUSTOMER_ID, LOYALTY_POINTS
FROM APP_CUSTOMERS WHERE CUSTOMER_ID = 1;
```

Expected: 195 points inside the transaction, then 185 after rollback. Keep transactions short and keep schema changes outside the transaction.

## 8. Practice and validation

1. Query customer 1 by its primary key and orders by the indexed status column.
2. Inspect `SHOW INDEXES IN SCHEMA HybridTable` and compare with Query Profile. An index's existence does not guarantee a particular plan for a tiny dataset.
3. Try a duplicate email in a separate statement and observe unique-key enforcement.
4. Explain why the parent customer must exist before its order can be inserted.

```sql
SELECT COUNT(*) AS CUSTOMER_COUNT FROM APP_CUSTOMERS;
SELECT COUNT(*) AS ORDER_COUNT FROM APP_ORDERS;
SELECT CUSTOMER_ID, LOYALTY_POINTS
FROM APP_CUSTOMERS WHERE CUSTOMER_ID = 1;
```

Expected counts: three customers and two orders; customer 1 has 185 points.

## 9. Optional cleanup

Use the original database and the same role that created the objects. Drop the child before the parent.

```sql
USE SCHEMA HybridTable;
DROP TABLE IF EXISTS APP_ORDERS;
DROP TABLE IF EXISTS APP_CUSTOMERS;
DROP SCHEMA IF EXISTS HybridTable RESTRICT;
```

`RESTRICT` leaves a schema containing other objects in place. The existing database and `SNOWFLAKE_LEARNING_WH` are retained.